<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_06_latency.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->


# Ex_12.2 · 06 — Latency, Measured Honestly on Colab

**Deep Learning for Engineering · Aalborg University**

The lecture argued that a learned screener is worth having because it answers
many contingencies at once. That is a claim about **latency**, and a claim about
latency has to be measured.

You do not have a Jetson Thor. This notebook does not pretend otherwise.

**What you can measure here, and what you cannot.** Colab gives you a T4 or an
A100 — different silicon from the Thor, different memory bandwidth, different
power envelope. An absolute millisecond figure measured here says nothing about
what the board would do.

What *does* carry across hardware is the **shape**: how cost scales with batch
size, what half precision is worth as a ratio, how cost grows with the graph,
and how much of the total is fixed overhead rather than per-contingency work.
Those ratios come from the model and the arithmetic, not from the device.

So you will report **relative numbers with confidence, and absolute numbers as
pending**. The mini-project in L13 runs the same script on a real Thor and fills
the empty column.

---

### The rule that makes any of this meaningful

A single timing on Colab is noise. The GPU is shared, clocks vary, and the first
call includes compilation. Every number below is a **median of repeated runs
after a warm-up**, and you report the spread alongside it.


In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import time, json, statistics, platform
import numpy as np
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", DEV)
if DEV == "cuda":
    print("gpu    :", torch.cuda.get_device_name(0))
    print("memory : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1e9))
print("torch  :", torch.__version__)

def timed(fn, warmup=10, runs=50):
    """Median wall time in ms, plus the interquartile spread.
    Warm-up matters: the first calls include kernel compilation."""
    for _ in range(warmup):
        fn()
    if DEV == "cuda": torch.cuda.synchronize()
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        if DEV == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1e3)
    ts.sort()
    q1, q3 = ts[len(ts)//4], ts[3*len(ts)//4]
    return statistics.median(ts), q3 - q1

## 1 · The model you are timing

Reuse the graph network from notebook 03. If you have a trained checkpoint, load
it; otherwise an untrained model of the same shape gives the same timings, since
latency depends on the architecture rather than on the weights.


In [ ]:
import torch.nn as nn

class MPLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.msg = nn.Sequential(nn.Linear(2*d, d), nn.Tanh())
        self.upd = nn.Sequential(nn.Linear(2*d, d), nn.Tanh())
    def forward(self, h, A):
        # A is a dense adjacency, (N, N); h is (B, N, d)
        nb = torch.einsum("ij,bjd->bid", A, h)
        m  = self.msg(torch.cat([h, nb], dim=-1))
        return self.upd(torch.cat([h, m], dim=-1))

class Screener(nn.Module):
    def __init__(self, d=64, layers=4, f_in=6):
        super().__init__()
        self.enc = nn.Linear(f_in, d)
        self.mp  = nn.ModuleList([MPLayer(d) for _ in range(layers)])
        self.out = nn.Linear(d, 1)
    def forward(self, x, A):
        h = self.enc(x)
        for l in self.mp: h = l(h, A)
        return self.out(h).squeeze(-1)

model = Screener().to(DEV).eval()
print(sum(p.numel() for p in model.parameters()), "parameters")

## 2 · Scaling with batch size

Screening is embarrassingly parallel: every contingency is an independent
forward pass, so they batch. This is the measurement that matters most, because
it is the reason a learned screener is worth having at all.

Plot cost per contingency against batch size. It should fall steeply and then
flatten — the flattening point is where you stop being overhead-bound.


In [ ]:
N = 6                      # buses in your test case
A = torch.zeros(N, N, device=DEV)
for i, j in [(0,1),(1,2),(2,3),(0,4),(2,5)]:
    A[i,j] = A[j,i] = 1.0
A = A / A.sum(1, keepdim=True).clamp(min=1)     # normalised, as in notebook 03

batches = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
per_case = []
for B in batches:
    x = torch.randn(B, N, 6, device=DEV)
    with torch.no_grad():
        med, iqr = timed(lambda: model(x, A))
    per_case.append(med / B)
    print(f"  batch {B:>4}: {med:7.3f} ms total, {med/B:7.4f} ms per contingency  (iqr {iqr:.3f})")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,3.2))
ax.loglog(batches, per_case, "o-", lw=2)
ax.set_xlabel("batch size (contingencies at once)")
ax.set_ylabel("ms per contingency")
ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

speedup = per_case[0] / per_case[-1]
print(f"batching {batches[-1]} at once is {speedup:.1f}x cheaper per contingency than one at a time")

## 3 · What half precision is worth

A ratio, not a number. Both this and the Thor will gain from lower precision;
how much differs, but the direction and rough magnitude carry across.

**Check the answers, not only the clock.** A speedup that changes your verdicts
is not a speedup. Report the largest change in predicted margin alongside the
timing.


In [ ]:
B = 128
x = torch.randn(B, N, 6, device=DEV)

with torch.no_grad():
    med32, _ = timed(lambda: model(x, A))
    y32 = model(x, A).float()

if DEV == "cuda":
    m16 = Screener().to(DEV).eval().half()
    m16.load_state_dict({k: v.half() for k, v in model.state_dict().items()})
    x16, A16 = x.half(), A.half()
    with torch.no_grad():
        med16, _ = timed(lambda: m16(x16, A16))
        y16 = m16(x16, A16).float()
    drift = (y32 - y16).abs().max().item()
    print(f"  fp32 {med32:.3f} ms   fp16 {med16:.3f} ms   ratio {med32/med16:.2f}x")
    print(f"  largest change in predicted margin: {drift:.2e}")
    print("  -> report the ratio; the absolute ms is this GPU's, not the Thor's")
else:
    print("  no GPU in this runtime; rerun with a GPU runtime for this section")

## 4 · How cost grows with the network

Your test case has six buses. A real area has hundreds. Message passing is
linear in edges per layer, so cost should grow close to linearly — measure it
rather than assuming it, and say which it was.


In [ ]:
sizes = [6, 12, 24, 48, 96, 192]
cost = []
for n in sizes:
    An = (torch.rand(n, n, device=DEV) < 0.25).float()
    An = torch.maximum(An, An.T)
    An.fill_diagonal_(0)
    An = An / An.sum(1, keepdim=True).clamp(min=1)
    xn = torch.randn(64, n, 6, device=DEV)
    with torch.no_grad():
        med, _ = timed(lambda: model(xn, An), warmup=5, runs=25)
    cost.append(med)
    print(f"  {n:>4} buses: {med:7.3f} ms for 64 contingencies")

fig, ax = plt.subplots(figsize=(7,3.0))
ax.plot(sizes, cost, "o-", lw=2)
ax.set_xlabel("buses"); ax.set_ylabel("ms per batch of 64")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5 · The comparison the lecture actually claims

Against what? Time-domain simulation on the same cases, from notebook 04. That
is the number that decides whether any of this was worth doing.

Both are measured on the same machine, so **this ratio is the one honest
absolute claim in the notebook** — it is a like-for-like comparison, and it does
not depend on which GPU you happened to get.


In [ ]:
# Fill in from notebook 04: seconds for the full contingency set by integration.
SIM_SECONDS_PER_CASE = None        # <- MEASURE in notebook 04

B = 256
x = torch.randn(B, N, 6, device=DEV)
with torch.no_grad():
    med, _ = timed(lambda: model(x, A))
learned_ms_per_case = med / B

print(f"  learned screener : {learned_ms_per_case:.4f} ms per contingency")
if SIM_SECONDS_PER_CASE:
    ratio = (SIM_SECONDS_PER_CASE * 1e3) / learned_ms_per_case
    print(f"  integration      : {SIM_SECONDS_PER_CASE*1e3:.1f} ms per contingency")
    print(f"  ratio            : {ratio:,.0f}x")
    print("  this ratio is like-for-like and is the claim worth reporting")
else:
    print("  set SIM_SECONDS_PER_CASE from notebook 04 to complete the comparison")

## 6 · The table you hand in

Two columns. One you filled in, one you could not.

| quantity | this runtime | Jetson Thor |
|---|---|---|
| ms per contingency, batch 1 | measured | **pending** |
| ms per contingency, batch 256 | measured | **pending** |
| batching speedup | measured — **carries across** | expect similar |
| fp16 ratio | measured — **carries across** | expect similar |
| scaling exponent in buses | measured — **carries across** | expect similar |
| ratio against integration | measured — **like-for-like** | expect similar |

**The middle four rows are your result.** They are properties of the model and
the arithmetic, and they would hold on any reasonable accelerator. The first two
are properties of the GPU Colab gave you today, and they are not transferable.

Do not write "the model runs in X ms on edge hardware." You have not shown that.
Write "batching 256 contingencies costs Nx less per case than one at a time, and
the absolute figure on target hardware is pending."

The L13 mini-project runs this notebook unchanged on a real Thor and fills the
right-hand column.

---

### Hand in

`Ex12.2b_<team>.json` with the six rows, the interquartile spread on every
timing, the GPU you were given, and one sentence on which numbers you would
defend to an operator and which you would not.


In [ ]:
submission = {
    "team": "team-01",                      # <- your team
    "runtime_gpu": torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu",
    "ms_per_case_batch1":   round(per_case[0], 5),
    "ms_per_case_batch512": round(per_case[-1], 5),
    "batching_speedup":     round(per_case[0]/per_case[-1], 2),
    "fp16_ratio":           None,           # <- from section 3
    "buses_vs_ms":          {str(s): round(c,3) for s, c in zip(sizes, cost)},
    "ratio_vs_integration": None,           # <- from section 5
    "which_i_would_defend": "",             # <- one sentence
}
with open(f"Ex12.2b_{submission['team']}.json", "w") as f:
    json.dump(submission, f, indent=2)
print(json.dumps(submission, indent=2))